In [4]:
import pandas as pd
import numpy as np
import re
import warnings

# Suppress deprecation warnings during pipeline execution
warnings.filterwarnings('ignore')

# Force full column visibility and standardize floating point output
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', lambda x: '%.3f' % x)


In [11]:
# Execute initial read into DataFrame from Excel
file_path = "../data/raw/emdat_climate_2000_2025.xlsx"

try:
    # Read the Excel file
    df = pd.read_excel(file_path)
    print(f"Ingestion successful. Matrix dimensions: {df.shape[0]} records, {df.shape[1]} features.")
    
    # Visual validation of the feature schema
    display(df.head(3))
    
except FileNotFoundError:
    print(f"[FATAL ERROR] Dataset missing at {file_path}. Verify directory paths.")
except ImportError:
    print("[ERROR] Pandas needs the 'openpyxl' library to read Excel files.")
    print("To fix this, create a new cell, run: !pip install openpyxl")

Ingestion successful. Matrix dimensions: 8905 records, 47 features.


,DisNo.,Historic,Classification Key,Disaster Group,Disaster Subgroup,Disaster Type,Disaster Subtype,External IDs,Event Name,ISO,Country,Subregion,Region,Location,Origin,Associated Types,OFDA/BHA Response,Appeal,Declaration,AID Contribution ('000 US$),Magnitude,Magnitude Scale,Latitude,Longitude,River Basin,Start Year,Start Month,Start Day,End Year,End Month,End Day,Total Deaths,No. Injured,No. Affected,No. Homeless,Total Affected,Reconstruction Costs ('000 US$),"Reconstruction Costs, Adjusted ('000 US$)",Insured Damage ('000 US$),"Insured Damage, Adjusted ('000 US$)",Total Damage ('000 US$),"Total Damage, Adjusted ('000 US$)",CPI,Admin Units,GADM Admin Units,Entry Date,Last Update
0,2018-0040-BRA,No,nat-hyd-flo-flo,Natural,Hydrological,Flood,Flood (General),DFO:4576,NaN,BRA,Brazil,Latin America and the Caribbean,Americas,Rio de Janeiro,Heavy rains,Collapse|Flood,No,No,No,NaN,55138.950,Km2,-22.479,-44.095,NaN,2018,2.000,14.000,2018,2.000,16.000,4.000,NaN,250.000,NaN,250.000,NaN,NaN,NaN,NaN,10000.000,12492.000,80.050,"[{""adm2_code"":9961,""adm2_name"":""Rio De Janeiro""}]","[{""gid_2"":""BRA.19.68_2"",""migration_date"":""2025...",2018-02-20,2025-12-20
1,2002-0351-USA,No,nat-cli-wil-for,Natural,Climatological,Wildfire,Forest fire,NaN,NaN,USA,United States of America,Northern America,Americas,Colorado province,NaN,NaN,No,No,No,NaN,770.000,Km2,NaN,NaN,NaN,2002,6.000,8.000,2002,6.000,8.000,NaN,NaN,1500.000,72.000,1572.000,NaN,NaN,NaN,NaN,20000.000,34879.000,57.342,"[{""adm1_code"":3219,""adm1_name"":""Colorado""}]","[{""gid_1"":""USA.6_1"",""migration_date"":""2025-12-...",2003-07-01,2025-12-20
2,2022-0770-RWA,No,nat-hyd-flo-flo,Natural,Hydrological,Flood,Flood (General),NaN,NaN,RWA,Rwanda,Sub-Saharan Africa,Africa,Kigali,NaN,NaN,No,No,No,NaN,NaN,Km2,NaN,NaN,NaN,2022,11.000,17.000,2022,11.000,18.000,3.000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,93.295,"[{""adm1_code"":21970,""adm1_name"":""Kigali City/U...","[{""gid_1"":""RWA.5_1"",""migration_date"":""2025-12-...",2022-11-25,2025-12-20


In [12]:
# Standardize EM-DAT's heterogeneous column naming conventions to snake_case.
def standardize_columns(columns):
    cleaned_cols = []
    for col in columns:
        col = str(col).lower()
        col = re.sub(r"\('000 us\$\)", "usd_k", col) 
        col = re.sub(r"\(us\$\)", "usd", col)
        col = re.sub(r"[^a-z0-9_]+", "_", col)
        col = col.strip("_")
        cleaned_cols.append(col)
    return cleaned_cols

df.columns = standardize_columns(df.columns)
print("Normalized Feature Space:")
print(df.columns.tolist())

Normalized Feature Space:
['disno', 'historic', 'classification_key', 'disaster_group', 'disaster_subgroup', 'disaster_type', 'disaster_subtype', 'external_ids', 'event_name', 'iso', 'country', 'subregion', 'region', 'location', 'origin', 'associated_types', 'ofda_bha_response', 'appeal', 'declaration', 'aid_contribution_usd_k', 'magnitude', 'magnitude_scale', 'latitude', 'longitude', 'river_basin', 'start_year', 'start_month', 'start_day', 'end_year', 'end_month', 'end_day', 'total_deaths', 'no_injured', 'no_affected', 'no_homeless', 'total_affected', 'reconstruction_costs_usd_k', 'reconstruction_costs_adjusted_usd_k', 'insured_damage_usd_k', 'insured_damage_adjusted_usd_k', 'total_damage_usd_k', 'total_damage_adjusted_usd_k', 'cpi', 'admin_units', 'gadm_admin_units', 'entry_date', 'last_update']


In [13]:
# Standardize the primary temporal anchor
year_col = 'start_year' if 'start_year' in df.columns else 'year'

if year_col in df.columns:
    initial_rows = len(df)
    df = df.dropna(subset=[year_col])
    
    # Coerce to integer to prevent float indexing errors
    df[year_col] = df[year_col].astype(int)
    
    dropped = initial_rows - len(df)
    print(f"Temporal anchor '{year_col}' cast to integer. Dropped {dropped} unanchored records.")
else:
    print(f"[WARNING] Temporal column '{year_col}' not found. Review feature list.")

Temporal anchor 'start_year' cast to integer. Dropped 0 unanchored records.


In [14]:
# Quantify the proportion of missing values across all vectors.
sparsity = df.isnull().sum() / len(df) * 100
sparsity_df = pd.DataFrame({
    'missing_count': df.isnull().sum(), 
    'missing_percentage': sparsity
})

# Filter for vectors containing nulls and sort by severity
sparsity_df = sparsity_df[sparsity_df['missing_count'] > 0].sort_values(by='missing_percentage', ascending=False)

print("Feature Sparsity Report (>0% Null):")
display(sparsity_df.head(15))

Feature Sparsity Report (>0% Null):


,missing_count,missing_percentage
reconstruction_costs_adjusted_usd_k,8885,99.775
reconstruction_costs_usd_k,8884,99.764
aid_contribution_usd_k,8510,95.564
insured_damage_adjusted_usd_k,8272,92.892
insured_damage_usd_k,8263,92.791
no_homeless,7872,88.400
latitude,7769,87.243
longitude,7769,87.243
river_basin,7597,85.312
event_name,7192,80.764


In [15]:
import os

# Define Stage 1 output artifact path
output_path = "../data/processed/01_emdat_base_normalization.csv"

# Dynamically instantiate the directory tree to prevent I/O exceptions
os.makedirs(os.path.dirname(output_path), exist_ok=True)

# Serialize intermediate DataFrame to disk, suppressing arbitrary Pandas index arrays
df.to_csv(output_path, index=False)
print(f"[I/O Status] Stage 1 artifact successfully serialized to: {output_path}")

[I/O Status] Stage 1 artifact successfully serialized to: ../data/processed/01_emdat_base_normalization.csv
